<h2> Imports, loading event-log function and cleaning pipeline </h2>

In [2]:
import numpy as np
import pandas as pd
import pm4py
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder
import statistics
from collections import Counter
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import CountVectorizer
from collections import defaultdict
from sklearn.model_selection import GridSearchCV
import itertools


In [3]:
# Importing dataset from file path
def import_xes(file_path):
    log = pm4py.read_xes(file_path)
    return pm4py.convert_to_dataframe(log)

# Cleaning dataset: removing unnecessary columns, shifting to resource focus
def clean_dataset(df):
    df_final = df[['case:concept:name', 'concept:name', 'org:resource', 'time:timestamp']]
    df_final = df_final.sort_values(['org:resource', 'time:timestamp'])
    return df_final


# creating 80/20 split based on resources, ensuring a resource is in EITHER the test set OR the train set
def create_split(df_clean, test_size):
    resource_traces = (
        df_clean.sort_values(["org:resource", "time:timestamp"])
               .groupby("org:resource")["concept:name"]
               .apply(list)
    )

    resource_traces = resource_traces[resource_traces.apply(len) > 1]

    resources = resource_traces.index.tolist()

    # create set of train/test resource ids
    train_res, test_res = train_test_split(
        resources,
        test_size=test_size,
        random_state=1
    )

    train_traces = resource_traces.loc[train_res]
    test_traces = resource_traces.loc[test_res]

    return train_traces, test_traces


# prefix extraction on set list of prefix lengths, already implicitly buckets on prefix length
def build_prefix_df(resource_traces, prefix_lengths=[10], sliding_window=False):
    all_rows = []
    for length in prefix_lengths:
        for resource, seq in resource_traces.items():

            if(len(seq) < length + 1):
                continue

            if(sliding_window):
                for i in range(length, len(seq)):
                    prefix = seq[i-length:i]
                    next_act = seq[i]

                    all_rows.append({
                    'resource': resource,
                    'subtrace': prefix,
                    'prefix_length': length,
                    'last_activity': prefix[-1],
                    'next_activity': next_act
                    })
            else:
                prefix = seq[-(length+1):-1]
                next_act = seq[-1]

                all_rows.append({
                'resource': resource,
                'subtrace': prefix,
                'prefix_length': length,
                'last_activity': prefix[-1],
                'next_activity': next_act
                })
            
    return pd.DataFrame(all_rows)




def process_dataset(df, prefix_lengths):
    df_clean = clean_dataset(df)

    train_split, test_split = create_split(df_clean, 0.2)

    train_df = build_prefix_df(train_split, prefix_lengths, True)
    test_df = build_prefix_df(test_split, prefix_lengths, True)

    return train_df, test_df


<h1> Loading event-logs and transforming</h1>

<h4> Loading datasets </h4>

In [4]:
# df_2013, df_2013_prefixes = process_dataset("datasets/BPI_Challenge_2013_incidents.xes")
df_2013 = import_xes("datasets/BPI_Challenge_2013_incidents.xes")
#train_df_2013, test_df_2013 = process_dataset(df_2013)
GLOBAL_prefix_lengths = [10, 20, 30, 40, 50, 75, 100, 125, 150]
train_df_2013, test_df_2013 = process_dataset(df_2013, GLOBAL_prefix_lengths)

/Users/gijskoppenberg/Documents/UU Jaar 4/OZP/Resource-Centric-NAP/.venv/lib/python3.12/site-packages/pm4py/utils.py:992: UserWarning: Install the optional requirement `rustxes` to import/export files faster.
  warnings.warn("Install the optional requirement `rustxes` to import/export files faster.")
/Users/gijskoppenberg/Documents/UU Jaar 4/OZP/Resource-Centric-NAP/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
parsing log, completed traces :: 100%|██████████| 7554/7554 [00:02<00:00, 3566.07it/s]


In [5]:
print(len(train_df_2013))

273719


In [7]:
def train_RF(X_train, X_test, y_train, y_test):
    rf = RandomForestClassifier(n_estimators=100, random_state=1)
    rf.fit(X_train, y_train)

    y_pred = rf.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy is: {accuracy}")
    f1score = f1_score(y_test, y_pred, average='weighted')
    print(f"f1score = {f1score}")

    report = classification_report(y_test, y_pred, output_dict=True)
    print(report)
    return rf

In [9]:
# Fitting one hot encoder on the training data and applying on both train and test sets
def ohe_data(train_df, test_df, feature_cols=['last_activity']):
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

    encoder.fit(train_df[feature_cols])

    X_train = encoder.transform(train_df[feature_cols])
    X_test = encoder.transform(test_df[feature_cols])

    print(len(X_train))

    return X_train, X_test, encoder

In [ ]:
# Training random forest on un-bucketed 
print("Global OHE performance:")
X_train, X_test, ohe_encoder = ohe_data(train_df_2013, test_df_2013)
global_ohe_rf = train_RF(X_train, X_test, train_df_2013['next_activity'], test_df_2013['next_activity']) 


Global OHE performance:
273719
273719
59045
Accuracy is: 0.593174697264798
f1score = 0.536184662777818


/Users/gijskoppenberg/Documents/UU Jaar 4/OZP/Resource-Centric-NAP/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/gijskoppenberg/Documents/UU Jaar 4/OZP/Resource-Centric-NAP/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


{'Accepted': {'precision': 0.6634254729288976, 'recall': 0.8189069498263801, 'f1-score': 0.7330120047748823, 'support': 39742.0}, 'Completed': {'precision': 0.24817299028931825, 'recall': 0.23861776879391663, 'f1-score': 0.24330159976445187, 'support': 10389.0}, 'Queued': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 8914.0}, 'accuracy': 0.593174697264798, 'macro avg': {'precision': 0.30386615440607195, 'recall': 0.3525082395400989, 'f1-score': 0.32543786817977804, 'support': 59045.0}, 'weighted avg': {'precision': 0.4902044938818863, 'recall': 0.593174697264798, 'f1-score': 0.536184662777818, 'support': 59045.0}}


/Users/gijskoppenberg/Documents/UU Jaar 4/OZP/Resource-Centric-NAP/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [10]:
param_grid = {
    'n_estimators': [50, 100,200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'bootstrap': [True, False]
}

results = []


for length in GLOBAL_prefix_lengths:
    print(f"Searching for length {length}")
    current_train_df = train_df_2013[train_df_2013['prefix_length'] == length]
    current_test_df = test_df_2013[test_df_2013['prefix_length'] == length]

    current_X_train, current_X_test, current_encoder = ohe_data(current_train_df, current_test_df)

    current_y_train = current_train_df['next_activity']
    current_y_test = current_test_df['next_activity']

    rf = train_RF(current_X_train, current_X_test, current_y_train, current_y_test)

    grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, 
                               cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
    
    grid_search.fit(current_X_train, current_y_train)
    
    best_model = grid_search.best_estimator_
    current_y_pred = best_model.predict(current_X_test)
    
    acc = accuracy_score(current_y_test, current_y_pred)
    f1 = f1_score(current_y_test, current_y_pred, average='weighted')

    
    results.append({
        'length': length,
        'support': len(current_y_test),
        'accuracy': acc,
        'f1score': f1,
        'best_params': grid_search.best_params_
    })
total_samples = sum(r['support'] for r in results)
weighted_acc = sum(r['accuracy'] * r['support'] for r in results) / total_samples
weighted_f1 = sum(r['f1score'] * r['support'] for r in results) / total_samples

print("-" * 30)
print(f"Weighted Average Accuracy: {weighted_acc:.4f}")
print(f"Weighted Average F1score: {weighted_f1:.4f}")

    

Searching for length 10
552
Accuracy is: 0.7291666666666666
f1score = 0.6149598393574298
{'Accepted': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 10.0}, 'Completed': {'precision': 0.7291666666666666, 'recall': 1.0, 'f1-score': 0.8433734939759037, 'support': 105.0}, 'Queued': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 29.0}, 'accuracy': 0.7291666666666666, 'macro avg': {'precision': 0.24305555555555555, 'recall': 0.3333333333333333, 'f1-score': 0.28112449799196787, 'support': 144.0}, 'weighted avg': {'precision': 0.5316840277777778, 'recall': 0.7291666666666666, 'f1-score': 0.6149598393574298, 'support': 144.0}}
Fitting 5 folds for each of 36 candidates, totalling 180 fits


/Users/gijskoppenberg/Documents/UU Jaar 4/OZP/Resource-Centric-NAP/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/gijskoppenberg/Documents/UU Jaar 4/OZP/Resource-Centric-NAP/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/gijskoppenberg/Documents/UU Jaar 4/OZP/Resource-Centric-NAP/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no pre

Searching for length 20
389
Accuracy is: 0.7522935779816514
f1score = 0.6459484125078054
{'Accepted': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 9.0}, 'Completed': {'precision': 0.7522935779816514, 'recall': 1.0, 'f1-score': 0.8586387434554974, 'support': 82.0}, 'Queued': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 18.0}, 'accuracy': 0.7522935779816514, 'macro avg': {'precision': 0.2507645259938838, 'recall': 0.3333333333333333, 'f1-score': 0.2862129144851658, 'support': 109.0}, 'weighted avg': {'precision': 0.565945627472435, 'recall': 0.7522935779816514, 'f1-score': 0.6459484125078054, 'support': 109.0}}
Fitting 5 folds for each of 36 candidates, totalling 180 fits


/Users/gijskoppenberg/Documents/UU Jaar 4/OZP/Resource-Centric-NAP/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/gijskoppenberg/Documents/UU Jaar 4/OZP/Resource-Centric-NAP/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/gijskoppenberg/Documents/UU Jaar 4/OZP/Resource-Centric-NAP/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no pre

Searching for length 30
310
Accuracy is: 0.7674418604651163
f1score = 0.6664626682986536
{'Accepted': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 7.0}, 'Completed': {'precision': 0.7674418604651163, 'recall': 1.0, 'f1-score': 0.868421052631579, 'support': 66.0}, 'Queued': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 13.0}, 'accuracy': 0.7674418604651163, 'macro avg': {'precision': 0.2558139534883721, 'recall': 0.3333333333333333, 'f1-score': 0.2894736842105263, 'support': 86.0}, 'weighted avg': {'precision': 0.588967009194159, 'recall': 0.7674418604651163, 'f1-score': 0.6664626682986536, 'support': 86.0}}
Fitting 5 folds for each of 36 candidates, totalling 180 fits


...

Searching for length 125
115
Accuracy is: 0.8125
f1score = 0.728448275862069
{'Accepted': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 3.0}, 'Completed': {'precision': 0.8125, 'recall': 1.0, 'f1-score': 0.896551724137931, 'support': 26.0}, 'Queued': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 3.0}, 'accuracy': 0.8125, 'macro avg': {'precision': 0.2708333333333333, 'recall': 0.3333333333333333, 'f1-score': 0.2988505747126437, 'support': 32.0}, 'weighted avg': {'precision': 0.66015625, 'recall': 0.8125, 'f1-score': 0.728448275862069, 'support': 32.0}}
Fitting 5 folds for each of 36 candidates, totalling 180 fits


/Users/gijskoppenberg/Documents/UU Jaar 4/OZP/Resource-Centric-NAP/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/gijskoppenberg/Documents/UU Jaar 4/OZP/Resource-Centric-NAP/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/gijskoppenberg/Documents/UU Jaar 4/OZP/Resource-Centric-NAP/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no pre

Searching for length 150
92
Accuracy is: 0.8461538461538461
f1score = 0.7756410256410255
{'Accepted': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 2.0}, 'Completed': {'precision': 0.8461538461538461, 'recall': 1.0, 'f1-score': 0.9166666666666666, 'support': 22.0}, 'Queued': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 2.0}, 'accuracy': 0.8461538461538461, 'macro avg': {'precision': 0.28205128205128205, 'recall': 0.3333333333333333, 'f1-score': 0.3055555555555555, 'support': 26.0}, 'weighted avg': {'precision': 0.7159763313609468, 'recall': 0.8461538461538461, 'f1-score': 0.7756410256410255, 'support': 26.0}}
Fitting 5 folds for each of 36 candidates, totalling 180 fits


/Users/gijskoppenberg/Documents/UU Jaar 4/OZP/Resource-Centric-NAP/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/gijskoppenberg/Documents/UU Jaar 4/OZP/Resource-Centric-NAP/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/gijskoppenberg/Documents/UU Jaar 4/OZP/Resource-Centric-NAP/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no pre

------------------------------
Weighted Average Accuracy: 0.7789
Weighted Average F1score: 0.6869


<h3> Creating last-state encoding </h3>

<h3> Creating bigrams </h3>

In [ ]:
def create_bigram_features(df_train, df_test):

    # all activities in the train set
    train_activities = set([act for prefix in df_train['subtrace'] for act in prefix])
    unique_activities = sorted(list(train_activities))

    # all possible bigram transitions
    all_transitions = [f"{a}->{b}" for a in unique_activities for b in unique_activities]

    #print(f"Found {len(unique_activities)} unique activities.")
    print(f"Creating features for {len(all_transitions)} possible bigrams...")

    def _extract_counts(df):
        bigram_rows = []
        for prefix in df['subtrace']:
            counts = defaultdict(int)
            
            for i in range(len(prefix) - 1):
                key = f"{prefix[i]}->{prefix[i+1]}"
                counts[key] += 1
            
            row = [counts[t] for t in all_transitions]
            bigram_rows.append(row)
            
        return pd.DataFrame(bigram_rows, columns=all_transitions, index=df.index)

    X_train_bigram = _extract_counts(df_train)
    X_test_bigram = _extract_counts(df_test)
    
    return X_train_bigram, X_test_bigram

X_train2, X_test2 = create_bigram_features(train_df_2013, test_df_2013)
y_train2 = train_df_2013['next_activity']
y_test2 = test_df_2013['next_activity']

# Non-bucketed random forest on bigram encoded 
train_RF(X_train2, X_test2, y_train2, y_test2)

# Bucketed random forest on bigram encoded
for length in GLOBAL_prefix_lengths:
    print(f"Searching for length {length}")
    current_train_df = train_df_2013[train_df_2013['prefix_length'] == length]
    current_test_df = test_df_2013[test_df_2013['prefix_length'] == length]

    current_X_train, current_X_test = create_bigram_features(current_train_df, current_test_df)

    current_y_train = current_train_df['next_activity']
    current_y_test = current_test_df['next_activity']

    rf = train_RF(current_X_train, current_X_test, current_y_train, current_y_test)

    grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, 
                               cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
    
    grid_search.fit(current_X_train, current_y_train)
    
    best_model = grid_search.best_estimator_
    current_y_pred = best_model.predict(current_X_test)
    
    acc = accuracy_score(current_y_test, current_y_pred)
    f1 = f1_score(current_y_test, current_y_pred, average='weighted')

    
    results.append({
        'length': length,
        'support': len(current_y_test),
        'accuracy': acc,
        'f1_score': f1,
        'best_params': grid_search.best_params_
    })
total_samples = sum(r['support'] for r in results)
weighted_acc = sum(r['accuracy'] * r['support'] for r in results) / total_samples
weighted_f1 = sum(r['f1_score'] * r['support'] for r in results) / total_samples

print("-" * 30)
print(f"Weighted Average Accuracy: {weighted_acc:.4f}")
print(f"Weighted Average F1score: {weighted_f1:.4f}")
    





Creating features for 16 possible bigrams...
Accuracy is: 0.791869918699187
f1score = 0.7867346908760161
{'Accepted': {'precision': 0.4, 'recall': 0.2916666666666667, 'f1-score': 0.3373493975903614, 'support': 48.0}, 'Completed': {'precision': 0.8742393509127789, 'recall': 0.8941908713692946, 'f1-score': 0.884102564102564, 'support': 482.0}, 'Queued': {'precision': 0.4827586206896552, 'recall': 0.49411764705882355, 'f1-score': 0.4883720930232558, 'support': 85.0}, 'accuracy': 0.791869918699187, 'macro avg': {'precision': 0.5856659905341447, 'recall': 0.5599917283649283, 'f1-score': 0.5699413515720604, 'support': 615.0}, 'weighted avg': {'precision': 0.7831184551196425, 'recall': 0.791869918699187, 'f1-score': 0.7867346908760161, 'support': 615.0}}
Searching for length 10
Creating features for 16 possible bigrams...
Accuracy is: 0.75
f1score = 0.7516426282051282
{'Accepted': {'precision': 0.3, 'recall': 0.3, 'f1-score': 0.3, 'support': 10.0}, 'Completed': {'precision': 0.844660194174757

KeyError: 'f1_score'

In [ ]:
import numpy as np
from gensim.models import Word2Vec
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

# Train word2vec
word2vec_model = Word2Vec(sentences=train_df_2013['subtrace'], vector_size=16, window=5, min_count=1, sg=1)

def word2vec_embed_data(w2v_model, df, window_size=3):
    X = []
    y = df['next_activity'].values
    for trace in df['subtrace']:
        current_window = trace[-window_size:]

        vectors = []
        for act in current_window:
            if act in w2v_model.wv:
                vectors.append(w2v_model.wv[act])
            else:
                vectors.append(np.zeros(w2v_model.vector_size))
        
        while len(vectors) < window_size:
            vectors.insert(0, np.zeros(w2v_model.vector_size))
        X.append(vectors)
    X = np.array(X)
    X_flat = X.reshape(X.shape[0], -1)

    return X_flat, y

train_X, train_Y_raw = word2vec_embed_data(word2vec_model, train_df_2013)
test_X, test_Y_raw = word2vec_embed_data(word2vec_model, test_df_2013)

le = LabelEncoder()

train_Y = le.fit_transform(train_Y_raw)

rf = RandomForestClassifier(n_estimators=100)
rf.fit(train_X, train_Y)

print("Training Complete. Input shape:", train_X.shape)

y_pred_encoded = rf.predict(test_X)
y_pred_names = le.inverse_transform(y_pred_encoded)

acc = accuracy_score(test_Y_raw, y_pred_names)
print(acc)

param_grid = {
    'vector_size': [8,16,32],
    'window': [3,5],
    'sg': [0,1], # 0=CBOW, 1=Skip-gram
}

# generate all combinations by hand, since gensim is not compatible with cv gridsearch
keys,values = zip(*param_grid.items())
combinations = [dict(zip(keys,v)) for v in itertools.product(*values)]

best_acc = 0
best_params = {}
best_length = 0

for length in GLOBAL_prefix_lengths:
    current_train_df = train_df_2013[train_df_2013['prefix_length'] == length]
    current_test_df = test_df_2013[test_df_2013['prefix_length'] == length]

    for params in combinations:
        # retraining model for specific params
        model = Word2Vec(
            sentences=current_train_df['subtrace'],
            vector_size=params['vector_size'],
            window=params['window'],
            min_count=1,
            sg=params['sg'],
            seed=1
        )

        current_X_train, current_y_train = word2vec_embed_data(model, current_train_df, params['window'])
        current_X_test, current_y_test = word2vec_embed_data(model, current_test_df,params['window'])

        current_y_train_encoded = le.transform(current_y_train)

        rf = RandomForestClassifier(n_estimators=50, random_state=1, n_jobs=1)
        rf.fit(current_X_train, current_y_train_encoded)

        y_pred_encoded = rf.predict(current_X_test)
        y_pred = le.inverse_transform(y_pred_encoded)

        acc=accuracy_score(current_y_test, y_pred)
        print(f"Params: {params} | Accuracy: {acc}")

        if acc > best_acc:
            best_acc = acc
            best_params = params
            best_length = length

print("-" * 30)
print(f"BEST ACCURACY: {best_acc:.4f}")
print(f"BEST PARAMS: {best_params}")
print(f"Best prefix length: {best_length}")




Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Training Complete. Input shape: (2268, 48)
0.751219512195122
Params: {'vector_size': 8, 'window': 3, 'sg': 0} | Accuracy: 0.7708333333333334
Params: {'vector_size': 8, 'window': 3, 'sg': 1} | Accuracy: 0.7708333333333334
Params: {'vector_size': 8, 'window': 5, 'sg': 0} | Accuracy: 0.7152777777777778
Params: {'vector_size': 8, 'window': 5, 'sg': 1} | Accuracy: 0.7152777777777778
Params: {'vector_size': 16, 'window': 3, 'sg': 0} | Accuracy: 0.7708333333333334
Params: {'vector_size': 16, 'window': 3, 'sg': 1} | Accuracy: 0.7708333333333334
Params: {'vector_size': 16, 'window': 5, 'sg': 0} | Accuracy: 0.7222222222222222
Params: {'vector_size': 16, 'window': 5, 'sg': 1} | Accuracy: 0.7222222222222222
Params: {'vector_size': 32, 'window': 3, 'sg': 0} | Accuracy: 0.7708333333333334
Params: {'vector_size': 32, 'window': 3, 'sg': 1} | Accuracy: 0.7708333333333334
Params: {'vector_size': 32, 'window': 5, 'sg': 0} | Accuracy: 0.7152777777777778
Params: {'vector_size': 32, 'window': 5, 'sg': 1} | 

In [ ]:
from gensim.models.doc2vec import Doc2Vec, TaggedDocument


def prepare_tagged_data(df):
    return[TaggedDocument(words=row['subtrace'], tags=[str(i)])
            for i, row in df.iterrows()]


train_tagged = prepare_tagged_data(train_df_2013)

model = Doc2Vec(
    vector_size=64,
    window=5,
    min_count=1,
    workers=4,
    epochs=40
)

model.build_vocab(train_tagged)
print("vocab build")

model.train(train_tagged, total_examples=model.corpus_count, epochs=model.epochs)
print("model trained")

def get_vectors(model, tagged_docs):
    return np.array([model.dv[doc.tags[0]] for doc in tagged_docs])

def infer_test_vectors(model, subtrace_list):
    return np.array([model.infer_vector(subtrace) for subtrace in subtrace_list])


X_train = get_vectors(model, train_tagged)
X_test = infer_test_vectors(model, test_df_2013['subtrace'].tolist())
print("x train and test made")

y_train = train_df_2013['next_activity']
y_test = test_df_2013['next_activity']


rf = RandomForestClassifier(n_estimators=50, random_state=1, n_jobs=1)
rf.fit(X_train, y_train)
print("rf fit")

y_pred = rf.predict(X_test)

acc=accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc}")


vocab build
model trained
x train and test made
rf fit
Accuracy: 0.5745787111525108
